# SHIVA-LLM Objective 02 — Knowledge-Aware Offline RL
Reproducible Colab starter. **Important:** This is an observational logged-action baseline, not a validated tutoring policy. It does not claim counterfactual improvements. Upload an interaction CSV and optionally HyperKT out-of-fold state estimates.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib torch

In [ ]:
import os, random, json, numpy as np, pandas as pd, torch
from google.colab import files
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
print('Upload ASSISTments interaction CSV (and optionally HyperKT state CSV).')
uploaded=files.upload()
print(list(uploaded))

## Input contract
Interaction CSV requires `user_id`, `problem_id`, `correct`, `order_id`; optional `skill_id`, `difficulty`, `hint_count`. If using a separate HyperKT CSV, it must contain `user_id`, `order_id`, `kt_probability` with predictions computed strictly from prior interactions, ideally out-of-fold for training. The baseline action is the **observed next problem difficulty bin**, not a hint intervention.

In [ ]:
paths=[p for p in uploaded if p.lower().endswith('.csv')]
assert paths,'Upload at least one CSV.'
main_path=next((p for p in paths if 'assist' in p.lower() or 'interaction' in p.lower()),paths[0])
df=pd.read_csv(main_path,low_memory=False)
print('Columns:',df.columns.tolist())
rename={'student_id':'user_id','problem_id':'problem_id','correct':'correct','order_id':'order_id','skill_id':'skill_id','hint_count':'hint_count'}
df=df.rename(columns={k:v for k,v in rename.items() if k in df.columns and v not in df.columns})
required={'user_id','problem_id','correct','order_id'}
assert required.issubset(df.columns),f'Missing: {required-set(df.columns)}'
df=df.dropna(subset=list(required)).copy()
df['correct']=pd.to_numeric(df.correct,errors='coerce'); df=df[df.correct.isin([0,1])].copy()
df['order_id']=pd.to_numeric(df.order_id,errors='coerce');df=df.dropna(subset=['order_id'])
df=df.sort_values(['user_id','order_id'],kind='stable').reset_index(drop=True)
print('Interactions:',len(df),'Learners:',df.user_id.nunique())

## Learner-disjoint partition and causal state construction

In [ ]:
from sklearn.model_selection import train_test_split
users=df.user_id.drop_duplicates().to_numpy()
assert len(users)>=10,'Need at least 10 learners for this demonstration.'
train_u,temp_u=train_test_split(users,test_size=.30,random_state=SEED)
val_u,test_u=train_test_split(temp_u,test_size=.50,random_state=SEED)
split_map={u:'train' for u in train_u}|{u:'val' for u in val_u}|{u:'test' for u in test_u}
df['split']=df.user_id.map(split_map)
# Previous correctness only: no current/future outcome in state.
g=df.groupby('user_id',sort=False)
df['prior_attempts']=g.cumcount()
df['prior_successes']=g.correct.cumsum()-df.correct
df['prior_accuracy']=(df.prior_successes+1)/(df.prior_attempts+2)
# Problem difficulty estimated from TRAIN interactions only; note this is descriptive, not an intervention.
train=df[df.split=='train']; problem_rate=train.groupby('problem_id').correct.agg(['sum','count'])
problem_rate['difficulty']=1-(problem_rate['sum']+1)/(problem_rate['count']+2)
df['difficulty']=df.problem_id.map(problem_rate.difficulty).fillna(.5)
# Optional external HyperKT predictions; must be prior-only and leakage-audited.
kt_paths=[p for p in paths if p!=main_path and 'hyperkt' in p.lower()]
if kt_paths:
    kt=pd.read_csv(kt_paths[0]); assert {'user_id','order_id','kt_probability'}.issubset(kt.columns)
    assert not kt.duplicated(['user_id','order_id']).any()
    df=df.merge(kt[['user_id','order_id','kt_probability']],on=['user_id','order_id'],how='left',validate='one_to_one')
    assert df.kt_probability.notna().all(),'Missing HyperKT predictions.'
    df['knowledge_state']=df.kt_probability.clip(0,1)
    print('Using uploaded HyperKT states; verify causal and out-of-fold provenance.')
else:
    df['knowledge_state']=df.prior_accuracy
    print('WARNING: using smoothed prior accuracy proxy, NOT HyperKT.')

## Construct logged transitions
Action = difficulty tercile of the **next observed problem**, estimated from training data. Reward is a transparent *proxy* for next-attempt correctness minus a mismatch penalty. This is not a measured learning gain. Do not treat next-question assignment as randomized.

In [ ]:
q=np.quantile(df.loc[df.split=='train','difficulty'],[1/3,2/3])
q=np.unique(q)
assert len(q)==2,'Difficulty quantiles collapsed; use a larger or more varied dataset.'
df['action']=np.digitize(df.difficulty,q).astype(int)
df['next_correct']=df.groupby('user_id').correct.shift(-1)
df['next_knowledge']=df.groupby('user_id').knowledge_state.shift(-1)
df['next_prior_accuracy']=df.groupby('user_id').prior_accuracy.shift(-1)
df['next_prior_attempts']=df.groupby('user_id').prior_attempts.shift(-1)
df['next_action']=df.groupby('user_id').action.shift(-1)
# Current state predicts the next action/outcome: shift action to align with observed next problem.
df['logged_action']=df.next_action
# Use state at current step; outcome from next interaction only.
df['reward']=df.next_correct-0.10*abs(df.logged_action-2*df.knowledge_state)/2
trans=df.dropna(subset=['logged_action','next_correct','next_knowledge']).copy()
trans['logged_action']=trans.logged_action.astype(int)
state_cols=['knowledge_state','prior_accuracy','prior_attempts']
trans['prior_attempts']=np.log1p(trans.prior_attempts)
trans['next_prior_attempts']=np.log1p(trans.next_prior_attempts)
print(trans.groupby('split').agg(n=('reward','size'),learners=('user_id','nunique'),mean_proxy_reward=('reward','mean')))
print('Action counts by split:');print(pd.crosstab(trans.split,trans.logged_action))

## Behavior cloning baseline (supported actions only)
This is the safe initial stage before offline RL. It estimates logged action propensity and tests generalization on held-out learners. It is **not** evidence that the selected policy improves outcomes.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report
tr=trans[trans.split=='train'];va=trans[trans.split=='val'];te=trans[trans.split=='test']
assert tr.logged_action.nunique()==3,'Training data must contain all three action bins.'
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=500,class_weight='balanced'))
model.fit(tr[state_cols],tr.logged_action)
for name,part in [('validation',va),('test',te)]:
    p=model.predict_proba(part[state_cols]);pred=model.classes_[p.argmax(axis=1)]
    print(name,'action accuracy',accuracy_score(part.logged_action,pred),'log loss',log_loss(part.logged_action,p,labels=model.classes_))
print(classification_report(te.logged_action,model.predict(te[state_cols]),zero_division=0))

## Logged outcome model (diagnostic, not causal)
Estimate E[observed reward | state, logged action]. Because question choice is confounded by learner and content, this model cannot identify the value of a new policy.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
features=state_cols+['logged_action']
outcome=HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=15,l2_regularization=5,random_state=SEED)
outcome.fit(tr[features],tr.reward)
for name,part in [('validation',va),('test',te)]:
    pred=outcome.predict(part[features]);print(name,'reward RMSE',np.sqrt(mean_squared_error(part.reward,pred)))
# Candidate action scores are diagnostic predictions, not proven action effects.
probe=te[state_cols].head(5).copy()
for a in range(3):
    x=probe.copy();x['logged_action']=a
    probe[f'predicted_proxy_reward_action_{a}']=outcome.predict(x[features])
display(probe)

## Export reproducibility artifacts

In [ ]:
import joblib,zipfile
os.makedirs('objective02_artifacts',exist_ok=True)
joblib.dump(model,'objective02_artifacts/behavior_policy.joblib')
joblib.dump(outcome,'objective02_artifacts/observational_reward_model.joblib')
trans[['user_id','order_id','split','knowledge_state','prior_accuracy','prior_attempts','logged_action','reward']].to_csv('objective02_artifacts/transitions.csv',index=False)
with open('objective02_artifacts/README.txt','w') as f:
    f.write('SHIVA-LLM Objective 02 starter. Logged-action behavior cloning and observational reward diagnostics only. Not an evaluated RL policy. HyperKT provenance and action overlap must be audited.\n')
with zipfile.ZipFile('objective02_starter_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in os.listdir('objective02_artifacts'):z.write('objective02_artifacts/'+p)
files.download('objective02_starter_results.zip')

## Next research stage
1. Audit exact HyperKT state alignment and learner-disjoint split provenance. 2. Define feasible instructional actions and availability constraints. 3. Quantify behavior-policy overlap and reward sensitivity. 4. Implement conservative offline RL (e.g. discrete CQL) only after these checks. 5. Evaluate on held-out learners with defensible off-policy evaluation, uncertainty intervals and sensitivity analyses; observational data cannot establish causal policy gains without adequate assumptions or prospective validation.